# FedTune runner (Kaggle)

**Before running:** right sidebar → *Session options* → **Accelerator = GPU**, **Internet = On**.
Add the dataset `fedtune-code` (and, for `MODE = "grid"`, the dataset `fedtune-tuned`).
Edit **only the next cell**, then use **Save Version → Save & Run All (Commit)** so it runs in the background.

In [ ]:
# ================= EDIT ONLY THIS CELL =================
MODE = "tune"             # "tune"  = find best hyper-parameters for ONE strategy
                          # "grid"  = run your shard of the final 108-run grid
TUNE_STRATEGY = "fedavg"  # only for MODE="tune": fedavg | diversity | clustered | hybrid
SHARD = "0/5"             # only for MODE="grid": "i/n"  (i = your shard number, n = total shards; same n for everyone)
TIME_BUDGET_H = 10.0      # stop cleanly after this many hours (Kaggle sessions are capped at about 12 h)
# =======================================================

In [ ]:
import hashlib, os, re, shutil, site, subprocess, sys, time, urllib.request, zipfile
from pathlib import Path
from tqdm.auto import tqdm

KAGGLE_INPUT = Path(os.environ.get("FT_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("FT_WORK", "/kaggle/working"))
CODE = Path(os.environ.get("FT_CODE", "/tmp/ft_code"))
TUNE_CONFIG = os.environ.get("FT_TUNE_CONFIG", "configs/tune.yaml")
REPO = CODE / "fedtune"
PY = sys.executable

def walk(root, maxdepth=4):
    root = Path(root); base = len(root.parts)
    for dp, dns, fns in os.walk(root):
        if len(Path(dp).parts) - base >= maxdepth:
            dns[:] = []
        yield Path(dp), dns, fns

def sh(cmd, quiet=False):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, executable="/bin/bash")
    output = []
    for line in p.stdout:
        output.append(line)
        if not quiet:
            print(line, end="", flush=True)
    rc = p.wait()
    if rc != 0 and quiet:
        print("".join(output), flush=True)
    return rc

assert MODE in ("tune", "grid"), 'MODE must be "tune" or "grid"'
if MODE == "tune":
    assert TUNE_STRATEGY in ("fedavg", "diversity", "clustered", "hybrid"), "unknown TUNE_STRATEGY"
    TAG = f"tune_{TUNE_STRATEGY}"
else:
    assert re.fullmatch(r"\d+/\d+", SHARD), 'SHARD must look like "2/5"'
    _i, _n = map(int, SHARD.split("/"))
    assert 0 <= _i < _n, "shard number must satisfy 0 <= i < n"
    TAG = "grid_" + SHARD.replace("/", "of")

setup_bar = tqdm(total=6, desc="Setting up FedTune", unit="step")

# ---- 1. find the code ----
setup_bar.set_postfix_str("Locating code...")
src = None
for dp, _, fns in walk(KAGGLE_INPUT):
    if "pyproject.toml" in fns and 'name = "fedtune"' in (dp / "pyproject.toml").read_text():
        src = dp
        break
if src is None:
    zips = [dp / f for dp, _, fns in walk(KAGGLE_INPUT) for f in fns if f.endswith(".zip")]
    if not zips:
        raise SystemExit("Cannot find the code. Add the dataset 'fedtune-code' via Add Input.")
    shutil.rmtree(CODE / "x", ignore_errors=True)
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(CODE / "x")
    for dp, _, fns in walk(CODE / "x"):
        if "pyproject.toml" in fns and 'name = "fedtune"' in (dp / "pyproject.toml").read_text():
            src = dp
            break
    if src is None:
        src = CODE / "x" / "fedtune"
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(src, REPO)
setup_bar.update(1)

# ---- 2. fingerprint ----
setup_bar.set_postfix_str("Computing fingerprint...")
h = hashlib.sha256()
for f in sorted(list((REPO / "fedtune").rglob("*.py")) + list((REPO / "configs").glob("*.yaml"))):
    h.update(str(f.relative_to(REPO)).encode()); h.update(f.read_bytes())
FINGERPRINT = h.hexdigest()[:16]
print("code fingerprint:", FINGERPRINT)
setup_bar.update(1)

# ---- 3. GPU + internet checks ----
setup_bar.set_postfix_str("Checking GPU & Internet...")
import torch
has_gpu = torch.cuda.is_available()
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if has_gpu else "NONE")
if not has_gpu and not os.environ.get("FT_ALLOW_CPU"):
    raise SystemExit("GPU is OFF. Right sidebar -> Session options -> Accelerator -> GPU, then run again.")
if not os.environ.get("FT_SKIP_NET"):
    try:
        urllib.request.urlopen("https://huggingface.co", timeout=20)
        print("internet: OK")
    except Exception as e:
        raise SystemExit(f"Internet is OFF: {e}")
setup_bar.update(1)

# ---- 4. install (bypass PEP 517 isolation) ----
setup_bar.set_postfix_str("Installing fedtune & dependencies...")
rc = sh(f'"{PY}" -m pip install --no-build-isolation --no-deps -e "{REPO}"', quiet=True)
if rc != 0:
    pth_file = Path(site.getsitepackages()[0]) / "fedtune.pth"
    pth_file.write_text(f"{REPO}\n")
    os.environ["PYTHONPATH"] = f"{REPO}:{os.environ.get('PYTHONPATH', '')}"
    if str(REPO) not in sys.path:
        sys.path.insert(0, str(REPO))

# Explicitly upgrade torchao to fix the version check crash
deps = [("peft", "peft"), ("scipy", "scipy"), ("yaml", "pyyaml"),
        ("safetensors", "safetensors"), ("pyarrow", "pyarrow"), ("torchao", "torchao>=0.16.0")]
for mod, pipname in tqdm(deps, desc="Checking packages", leave=False):
    if pipname.startswith("torchao"):
        sh(f'"{PY}" -m pip install -q --upgrade "{pipname}"')
    else:
        try:
            __import__(mod)
        except ImportError:
            assert sh(f'"{PY}" -m pip install -q {pipname}') == 0, f"Failed to install {pipname}"
setup_bar.update(1)

# ---- 5. resume support ----
setup_bar.set_postfix_str("Restoring previous runs...")
(WORK / "runs").mkdir(parents=True, exist_ok=True)
restored = 0
for dp, dns, fns in walk(KAGGLE_INPUT):
    if dp.name == "runs":
        for child in dp.iterdir():
            if child.is_dir():
                shutil.copytree(child, WORK / "runs" / child.name, dirs_exist_ok=True)
                restored += 1
        dns[:] = []
print("runs restored from an earlier session:", restored)
setup_bar.update(1)

# ---- 6. grid mode & validation ----
setup_bar.set_postfix_str("Validating config...")
CFG = TUNE_CONFIG
if MODE == "grid":
    found = [dp / f for dp, _, fns in walk(KAGGLE_INPUT) for f in fns if f == "tuned.yaml"]
    if len(found) != 1:
        raise SystemExit(f"Expected exactly one tuned.yaml, found {len(found)}.")
    shutil.copy(found[0], REPO / "tuned.yaml")
    CFG = "tuned.yaml"

assert sh(f'cd "{REPO}" && "{PY}" -m fedtune.cli validate -c {CFG}') == 0, "config does not validate"
setup_bar.update(1)
setup_bar.set_postfix_str("Done!")
setup_bar.close()

In [ ]:
# ---- how much work is this? ----
if MODE == "grid":
    (REPO / "_shard_info.py").write_text(
        "import sys\n"
        "from fedtune.config import load_config\n"
        "from fedtune.experiment import grid_cells\n"
        "cfg = load_config('tuned.yaml'); i, n = map(int, sys.argv[1].split('/'))\n"
        "allc = grid_cells(cfg); mine = allc[i::n]\n"
        "print(f'this shard: {len(mine)} of {len(allc)} runs')\n"
        "from collections import Counter\n"
        "print('strategies in this shard:', dict(Counter(c[3] for c in mine)))\n")
    sh(f'cd "{REPO}" && "{PY}" _shard_info.py {SHARD}')
else:
    print(f"tuning {TUNE_STRATEGY}: the trial count comes from tune.trials in the config (6 by default)")

In [ ]:
# ---- THE LONG RUN: progress bar + status line that update on a timer (not only when a log line appears) ----
import json, os, subprocess, sys, time
from pathlib import Path
from tqdm.auto import tqdm

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from fedtune.config import load_config
from fedtune import experiment as E

cfg_obj = load_config(str(REPO / CFG))                       # a plain dict
ROUNDS = cfg_obj["experiment"]["rounds"]
if MODE == "tune":
    EXPECTED = len(E.tuning_trials(cfg_obj, TUNE_STRATEGY))
else:
    _i, _n = map(int, SHARD.split("/"))
    EXPECTED = len(E.grid_cells(cfg_obj)[_i::_n])
print(f"to do in this notebook: {EXPECTED} runs x {ROUNDS} rounds = {EXPECTED * ROUNDS} rounds", flush=True)

RUNS = WORK / "runs"

def scan():
    """(completed runs, rounds logged so far, text about the run that is being worked on right now)"""
    done, rounds_logged = 0, 0
    for s in RUNS.glob("*/run_summary.json"):
        try:
            done += json.loads(s.read_text()).get("status") == "completed"
        except Exception:
            pass
    files = list(RUNS.glob("*/rounds.jsonl"))
    for f in files:
        try:
            rounds_logged += min(ROUNDS, sum(1 for _ in open(f)))
        except Exception:
            pass
    cur = "starting up (loading data / model)"
    if files:
        f = max(files, key=lambda p: p.stat().st_mtime)
        try:
            rec = json.loads(f.read_text().splitlines()[-1])
            acc = (rec.get("val_metrics") or {}).get("accuracy")
            spr = float(rec.get("round_time_s") or 0)
            left = max(ROUNDS - rec["round"], 0) * spr / 60
            cur = (f"{f.parent.name.replace('agnews-', '')}: round {rec['round']}/{ROUNDS}"
                   + (f", val_acc {acc:.3f}" if acc is not None else "")
                   + f", {spr:.0f}s/round, ~{left:.0f} min left in this run")
        except Exception:
            pass
    return done, rounds_logged, cur

budget = int(TIME_BUDGET_H * 3600)
out = f"experiment.output_dir={WORK}/runs"
if MODE == "tune":
    args = f'tune -c {TUNE_CONFIG} --set {out} "experiment.strategy=[{TUNE_STRATEGY}]"'
else:
    args = f"grid -c tuned.yaml --set {out} --shard {SHARD}"

POLL = float(os.environ.get("FT_POLL", "20"))      # seconds between bar updates
logp = WORK / "train.log"
d0, r0, _ = scan()
bar = tqdm(total=EXPECTED * ROUNDS, initial=min(r0, EXPECTED * ROUNDS), desc=f"FedTune ({TAG})", unit="round")
cmd = f'cd "{REPO}" && timeout {budget} "{PY}" -u -m fedtune.cli {args}'
env = {**os.environ, "PYTHONUNBUFFERED": "1"}          # so the training's own messages appear immediately

t0, pos, last_text = time.time(), 0, 0.0
with open(logp, "wb") as lf:
    p = subprocess.Popen(cmd, shell=True, stdout=lf, stderr=subprocess.STDOUT, executable="/bin/bash", env=env)
    while True:
        try:
            rc = p.wait(timeout=POLL)
        except subprocess.TimeoutExpired:
            rc = None
        with open(logp, "rb") as r:
            r.seek(pos); new = r.read(); pos = r.tell()
        if new:
            for ln in new.decode(errors="replace").splitlines():
                if "Loading weights" in ln or not ln.strip():      # drop the noisy per-model loading bars
                    continue
                tqdm.write(ln)
        done, rl, cur = scan()
        bar.n = min(rl, bar.total)
        bar.set_postfix_str(f"runs {done}/{EXPECTED} | {cur}", refresh=True)
        if time.time() - last_text > 300 or rc is not None:    # plain-text line too (commit logs don't redraw bars)
            tqdm.write(f"[{(time.time() - t0) / 60:6.1f} min] runs completed {done}/{EXPECTED} | {cur}")
            last_text = time.time()
        if rc is not None:
            break
bar.close()

print(f"\nfinished in {(time.time() - t0) / 3600:.2f} h, exit code {rc}")
if rc == 124:
    print("TIME BUDGET HIT. Not an error: run this notebook again with this notebook's output attached "
          "(Add Input -> Notebook Output) and it resumes where it stopped.")
for s in sorted(RUNS.glob("*/run_summary.json")):          # say WHY any run failed, instead of staying silent
    try:
        j = json.loads(s.read_text())
        if j.get("status") == "failed":
            print("FAILED:", s.parent.name, "->", j.get("reason"))
    except Exception:
        pass

In [ ]:
# ---- package results (this zip is what you send back) ----
import json
status = {}
for d in sorted((WORK / "runs").glob("*")):
    s = d / "run_summary.json"
    if s.exists():
        status[d.name] = json.loads(s.read_text()).get("status")
    elif any(p.is_file() for p in d.rglob("*")):
        status[d.name] = "interrupted"       # started, has checkpoints/logs, can be resumed
    else:
        status[d.name] = "untouched"         # empty folder fedtune creates for other shards' cells: ignored
ok = [k for k, v in status.items() if v == "completed"]
part = [k for k, v in status.items() if v in ("interrupted", "failed")]
print(f"completed runs: {len(ok)} | interrupted/failed (re-run to resume): {part}")

zpath = WORK / f"results_{TAG}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for top in ("runs", "tunes", "grids"):
        for f in (WORK / top).rglob("*") if (WORK / top).exists() else []:
            if not f.is_file():
                continue
            if top == "runs":
                st = status.get(f.relative_to(WORK / "runs").parts[0])
                if st == "completed" and "checkpoints" in f.parts:
                    continue   # finished runs don't need their checkpoints
            if top == "grids":
                continue       # per-shard summaries are partial; the merge step recomputes them
            z.write(f, f.relative_to(WORK))
(WORK / "FINGERPRINT.txt").write_text(f"{FINGERPRINT}\n")
print("wrote", zpath, f"({zpath.stat().st_size / 1e6:.1f} MB); code fingerprint {FINGERPRINT}")